# CA30 — Experiment & Reproducibility Template
This notebook is a lightweight, test-friendly experiment template for CA30. It demonstrates how to load configs, set deterministic seeds, run a tiny training loop, and save figures that are referenced by the report.

**NOTE:** Cells are intentionally small and deterministic; do not run heavy experiments here. Replace configs and expand experiments in your own runs.

In [ ]:
## 1) Environment setup & quick checks

# Shell commands (example, run in a terminal)
# python -m venv .venv && source .venv/bin/activate
# python -m pip install -r requirements.txt

# Python checks (run these cells to inspect versions):
import sys
import importlib
print("python:", sys.version.splitlines()[0])
for pkg in ("numpy", "yaml", "pytest"):
    try:
        m = importlib.import_module(pkg)
        print(pkg, m.__version__)
    except Exception:
        print(pkg, "not installed")

print('\nDry-run tests command (do not run heavy experiments here):')
print('pytest -q')

## 2) Run tests and inspect failures

# Quick way to run tests locally (don't run long jobs in CI):
# !pytest -q --maxfail=1 -k "not slow"

# If tests fail, inspect tests to find missing implementations. Example files:
# - tests/test_config.py
# - tests/test_model_forward.py
# - tests/test_imports.py

print('Run the pytest command above in your shell to get test output.')

In [ ]:
## 3) Demonstrate loading a config dataclass

from ca30.config import ExperimentConfig
from pathlib import Path

cfg = ExperimentConfig.load(Path("../configs/example.yaml").resolve())
print(cfg)

# Example: modify a field and save a new config for an experiment
cfg2 = ExperimentConfig(**{**cfg.__dict__, "seed": 999})
cfg2.save(Path("configs/example_override.yaml"))
print('Saved example_override.yaml')

In [ ]:
## 4) Utils and deterministic seeding

from ca30.utils import set_seed, ensure_dir

set_seed(int(cfg.seed))
print('seed set to', cfg.seed)

# ensure results dir exists
from pathlib import Path
out_dir = Path('results/demo_run')
ensure_dir(out_dir)
print('results will be saved to', out_dir.resolve())

In [ ]:
## 5) Model forward pass (lightweight test)

from ca30.model import BaseModel
import numpy as np

model = BaseModel(input_dim=cfg.input_dim, hidden_dim=cfg.hidden_dim, output_dim=cfg.output_dim)
print('Created model with backend:', model.backend)

x = np.zeros((2, cfg.input_dim), dtype=np.float32)
out = model.forward(x)
print('output shape:', out.shape)

# Assert shape is correct (example check)
assert out.shape == (2, cfg.output_dim)
print('Shape check passed')

In [ ]:
## 5) Instantiate model and run a forward pass

from ca30.model import BaseModel
import numpy as np

model = BaseModel(input_dim=cfg.input_dim, hidden_dim=cfg.hidden_dim, output_dim=cfg.output_dim)
# tiny dummy input
x = np.zeros((2, cfg.input_dim), dtype=np.float32)
out = model.forward(x)
print('forward output shape:', out.shape)

# simple assertion visible in the notebook
assert out.shape[0] == 2

In [ ]:
## 6) Lightweight training loop (dry-run)

from ca30.train import simple_train_epoch

history = {"loss": []}
for step in range(5):
    out = simple_train_epoch(model, batch_size=cfg.batch_size, input_dim=cfg.input_dim)
    # use mean squared output as a dummy loss for the curve
    loss = (out ** 2).mean()
    history["loss"].append(float(loss))

print('history:', history)

# Save tiny metrics
import json
from pathlib import Path
out_dir = Path('report/figures')
out_dir.mkdir(parents=True, exist_ok=True)
with (out_dir / 'metrics_demo.json').open('w') as f:
    json.dump(history, f)
print('Saved metrics to', out_dir / 'metrics_demo.json')

In [ ]:
## 7) Generate example figure and save to report/figures

import matplotlib.pyplot as plt
import json
from pathlib import Path

metrics_path = Path('report/figures/metrics_demo.json')
if metrics_path.exists():
    with metrics_path.open('r') as f:
        metrics = json.load(f)
else:
    metrics = {"loss": [0.1 * (1 + i) for i in range(5)]}

plt.figure(figsize=(4, 3))
plt.plot(metrics['loss'], marker='o')
plt.title('Demo learning curve')
plt.xlabel('step')
plt.ylabel('loss')
plt.tight_layout()
fig_path = Path('report/figures/figure1.png')
plt.savefig(fig_path, dpi=100)
plt.close()
print('Saved figure to', fig_path)
